# 02. Financial Statements — 재무제표 조회

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eddmpython/dartlab/blob/master/notebooks/tutorials/02_financial_statements.ipynb)

재무제표는 기업 분석의 출발점이다. DART 원본 재무제표를 조회하고, 데이터 구조를 이해하고, 원하는 항목을 추출하는 방법을 다룬다.

- 3대 재무제표 조회 (BS, IS, CF)
- 연결 vs 별도 재무제표
- 연도별 열 구조 이해
- 특정 계정항목 필터링
- 분기별 재무제표
- 원본 데이터의 한계와 주의사항

📖 [문서 보기](https://eddmpython.github.io/dartlab/docs/tutorials/financial-statements)

In [ ]:
!pip install -q dartlab

In [ ]:
import dartlab
import polars as pl

c = dartlab.Company("005930")

## 재무상태표 (Balance Sheet)

재무상태표는 **특정 시점**의 자산, 부채, 자본 잔액을 보여준다.

| 구분 | 의미 | 주요 항목 |
|------|------|----------|
| 자산 | 회사가 보유한 경제적 자원 | 유동자산, 비유동자산, 자산총계 |
| 부채 | 갚아야 할 의무 | 유동부채, 비유동부채, 부채총계 |
| 자본 | 자산 - 부채 | 자본금, 이익잉여금, 자본총계 |

핵심 등식: **자산 = 부채 + 자본**

In [ ]:
c.BS

In [ ]:
bs = c.BS
if bs is not None:
    print(bs.columns)
    total_assets = bs.filter(pl.col("account") == "자산총계")
    print(total_assets)

## 손익계산서 (Income Statement)

손익계산서는 **일정 기간** 동안의 수익과 비용을 보여준다.

| 구분 | 의미 | 주요 항목 |
|------|------|----------|
| 수익 | 영업활동으로 벌어들인 돈 | 매출액(수익) |
| 비용 | 수익 창출에 쓴 돈 | 매출원가, 판관비 |
| 이익 | 수익 - 비용 | 영업이익, 당기순이익 |

> 같은 "매출"이라도 회사마다 계정명이 다르다. 삼성전자는 "수익(매출액)", SK하이닉스는 "영업수익"이다. 이 문제는 [3. 시계열 분석](./03_timeseries.ipynb)에서 해결한다.

In [ ]:
c.IS

In [ ]:
is_df = c.IS
if is_df is not None:
    key_items = is_df.filter(
        pl.col("account").is_in(["매출액", "수익(매출액)", "영업이익"])
    )
    print(key_items)

## 현금흐름표 (Cash Flow Statement)

현금흐름표는 **일정 기간** 동안 실제로 현금이 얼마나 들어오고 나갔는지를 보여준다.

| 구분 | 의미 | 예시 |
|------|------|------|
| 영업활동 | 본업에서 벌어들인 현금 | 매출 수금, 급여 지급 |
| 투자활동 | 설비·자산 관련 현금 | 공장 건설, 장비 구매 |
| 재무활동 | 자금조달·상환 현금 | 차입, 배당 지급 |

In [ ]:
c.CF

In [ ]:
cf = c.CF
if cf is not None:
    cash_items = cf.filter(
        pl.col("account").is_in([
            "영업활동으로인한현금흐름",
            "투자활동으로인한현금흐름",
            "재무활동으로인한현금흐름"
        ])
    )
    print(cash_items)

## 연결 vs 별도

| 구분 | 포함 범위 | 보는 이유 |
|------|----------|----------|
| **연결** (CFS) | 모회사 + 자회사 전체 | 그룹 전체의 실질적 규모와 성과 |
| **별도** (OFS) | 모회사만 | 모회사 자체의 재무 상태 |

DartLab은 **연결 재무제표를 우선** 사용한다.

In [ ]:
# finance 시계열로 확인
c.finance.timeseries

## 열 구조 이해

반환되는 DataFrame의 열은 `account` + 연도들이다.

- `account`: 계정항목명 (한글)
- 연도 열: 해당 연도 결산 기준 금액 (원)

In [ ]:
bs = c.BS
print(bs.columns)

In [ ]:
row = bs.filter(pl.col("account") == "자산총계")
if row.height > 0:
    cols = [col for col in bs.columns if col != "account"]
    last_year = cols[-1]
    value = row[last_year][0]
    print(f"{last_year}년 자산총계: {value:,.0f} 원")
    print(f"                = {value/1e12:,.1f} 조원")

In [ ]:
key_accounts = ["자산총계", "유동자산", "비유동자산", "부채총계", "자본총계"]
summary = bs.filter(pl.col("account").is_in(key_accounts))
print(summary)

## 분기별 재무제표

| period 값 | 포함 보고서 | 열 형식 |
|-----------|------------|----------|
| `"y"` (기본값) | 사업보고서만 | `2020`, `2021`, ... |
| `"q"` | 1분기 + 반기 + 3분기 + 사업 | `2020Q1`, `2020Q2`, ... |
| `"h"` | 반기 + 사업 | `2020H1`, `2020H2`, ... |

> 분기별 손익계산서(IS)의 수치는 **누적값**이다. 독립 분기 매출이 필요하면 [3. 시계열 분석](./03_timeseries.ipynb)의 시계열 엔진을 사용한다.

In [ ]:
# 비율 시계열
c.finance.ratioSeries

In [ ]:
quarterly.IS

## 주의사항

### 계정명이 회사마다 다르다

같은 "매출"이라도 회사마다 다른 이름을 쓴다. `pl.col("account") == "매출액"`으로 필터링하면 일부 기업에서 빈 결과가 나올 수 있다. 이 문제는 시계열 엔진으로 해결한다.

### 금액 단위

DART 원본 데이터의 금액 단위는 **원(KRW)**이다.

### None 반환

데이터가 없는 기업은 `None`을 반환한다. 항상 `None` 체크를 해야 한다.

In [ ]:
bs = c.BS
if bs is not None:
    print(f"행 수: {bs.height}, 열 수: {bs.width}")
else:
    print("재무상태표 데이터 없음")

---

다음: [3. 시계열 분석](./03_timeseries.ipynb) — 계정 표준화, 분기별 독립 시계열, TTM